# 04 - Dataset Integration, Weather, Labeling, and Validation

This notebook is now a thin wrapper around the real pipeline stages. It keeps notebook traceability while using the same merge, weather, route-deviation, labeling, enrichment, and validation logic as `main.py`.

## 1. Setup and Controls

In [ ]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import main as pipeline_main
from src.data_validator import DataValidator
from src.utils import load_config

pipeline_main = importlib.reload(pipeline_main)

config = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
paths = config['paths']
processed_dir = PROJECT_ROOT / paths['processed_data_dir']

# These stages are much smaller than full ADS-B trajectory reconstruction, so forcing is usually fine here.
FORCE_MERGE_STAGE = True
FORCE_WEATHER_STAGE = True
FORCE_LABEL_STAGE = True
FORCE_VALIDATE_STAGE = True

validator = DataValidator(mode=config.get('validation_mode', 'warn_only'), log_dir='logs')

print(f'Project root: {PROJECT_ROOT}')
print(f'Processed data dir: {processed_dir}')

## 2. Input Check

In [ ]:
features_path = processed_dir / paths['features_file']
weather_path = pipeline_main._resolve_weather_input(paths)

if features_path.exists():
    df_features = pd.read_parquet(features_path)
    print(f'Trajectory features: {features_path} ({len(df_features):,} rows, {len(df_features.columns):,} columns)')
    display(df_features.head())
else:
    print(f'Missing trajectory features: {features_path}. Run notebook 03 first.')

print(f'Weather input candidate: {weather_path} ({"exists" if weather_path.exists() else "missing"})')

## 3. Merge ADS-B Features with Schedule Data

In [ ]:
pipeline_main.stage_merge(config, validator, force=FORCE_MERGE_STAGE)

merge_path = processed_dir / 'ml_dataset_merged.parquet'
if merge_path.exists():
    df_merged = pd.read_parquet(merge_path)
    print(f'Merged dataset: {len(df_merged):,} rows, {len(df_merged.columns):,} columns')
    if 'source_dataset' in df_merged.columns:
        display(df_merged['source_dataset'].value_counts(dropna=False).rename_axis('source_dataset').reset_index(name='rows'))
    display(df_merged.head())
else:
    print('Merged dataset was not created.')

## 4. Add Weather, Temporal, Airport, Congestion, and Route-Deviation Features

In [ ]:
pipeline_main.stage_weather(config, validator, force=FORCE_WEATHER_STAGE)

weather_output_path = processed_dir / 'ml_dataset_weather.parquet'
if weather_output_path.exists():
    df_weather = pd.read_parquet(weather_output_path)
    print(f'Weather-enriched dataset: {len(df_weather):,} rows, {len(df_weather.columns):,} columns')
    coverage_cols = [
        'weather_severity', 'weather_severity_dest', 'dep_hour', 'origin_flight_count',
        'origin_flights_1hr', 'route_stretch_ratio', 'lateral_deviation_mean_km',
    ]
    coverage = pd.DataFrame({
        'column': [c for c in coverage_cols if c in df_weather.columns],
        'non_null_pct': [round(float(df_weather[c].notna().mean() * 100), 2) for c in coverage_cols if c in df_weather.columns],
    })
    display(coverage)
    display(df_weather.head())
else:
    print('Weather-enriched dataset was not created.')

## 5. Generate Labels and Final ML Dataset

In [ ]:
pipeline_main.stage_label(config, validator, force=FORCE_LABEL_STAGE)

labeled_path = processed_dir / 'ml_dataset_labeled.parquet'
if labeled_path.exists():
    df_labeled = pd.read_parquet(labeled_path)
    print(f'Labeled dataset: {len(df_labeled):,} rows, {len(df_labeled.columns):,} columns')
    if 'label' in df_labeled.columns:
        display(df_labeled['label'].value_counts(dropna=False).rename_axis('label').reset_index(name='rows'))
    display(df_labeled.head())
else:
    print('Labeled dataset was not created.')

## 6. Validate and Sync Final Dataset

In [ ]:
pipeline_main.stage_validate(config, force=FORCE_VALIDATE_STAGE)

final_path = processed_dir / paths['ml_dataset_file']
if final_path.exists():
    df_final = pd.read_parquet(final_path)
    print(f'Final ML dataset: {len(df_final):,} rows, {len(df_final.columns):,} columns')
    if 'training_eligible' in df_final.columns:
        print(f'Training-eligible rows: {int(df_final["training_eligible"].fillna(False).sum()):,}')
    key_cols = [
        'label', 'delay_minutes', 'training_eligible', 'trajectory_quality_score',
        'is_full_flight', 'weather_severity', 'weather_severity_dest',
        'route_stretch_ratio', 'origin_delay_rate', 'dest_delay_rate',
    ]
    display(df_final[[c for c in key_cols if c in df_final.columns]].head())
else:
    print('Final ML dataset was not created.')